# imports

In [7]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, accuracy_score
import mlflow
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import mlflow.tensorflow
from tensorflow.keras.callbacks import EarlyStopping

# config

In [ ]:
ASSET = "BTC"
INTERVAL = "1h"

In [9]:
TIMESTEPS = 6

# mlflow

In [10]:
mlflow.set_tracking_uri("http://localhost:5000")

experiment_name = f"{ASSET}_{INTERVAL}_deep_learning"
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='/mlruns/13', creation_time=1778414378831, experiment_id='13', last_update_time=1778414378831, lifecycle_stage='active', name='BTC_1h_deep_learning', tags={}, trace_location=None, workspace='default'>

# load data

In [11]:
train_df = pd.read_parquet('../../../data/processed/train_btc_1h.parquet')
test_df = pd.read_parquet('../../../data/processed/test_btc_1h.parquet')

# splitting

In [12]:
features = [col for col in train_df.columns if col not in ['target_1h', 'target_direction']]
X_train = train_df[features]
y_train = train_df['target_direction']
X_test = test_df[features]
y_test = test_df['target_direction']

# checking the train data if scaled or not

In [13]:
print(X_train.describe().loc[['mean', 'std']].T.head(5))

                 mean       std
rsi_14  -7.601340e-16  1.000012
roc_10   7.302597e-18  1.000012
roc_20   1.029002e-17  1.000012
stoch_k  2.077921e-16  1.000012
stoch_d -4.461223e-16  1.000012


# LSTM

In [14]:
def create_sequences(X, y, time_steps=24):
    """
    Transforms 2D tabular data into 3D tensors for LSTM consumption.
    X: Numpy array of features
    y: Target series
    time_steps: historical periods to look back
    """
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y.iloc[i + time_steps])
        
    return np.array(Xs), np.array(ys)


X_train_seq, y_train_seq = create_sequences(X_train.values, y_train, TIMESTEPS)
X_test_seq, y_test_seq = create_sequences(X_test.values, y_test, TIMESTEPS)

print(f"Training Shape (Samples, TimeSteps, Features): {X_train_seq.shape}")
print(f"Testing Shape: {X_test_seq.shape}")

Training Shape (Samples, TimeSteps, Features): (42806, 6, 29)
Testing Shape: (10698, 6, 29)


# model

In [15]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(TIMESTEPS, X_train_seq.shape[2])),
    Dropout(0.4),
    
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    
    Dense(16, activation='relu'), Dropout(0.2,),
    
    Dense(1, activation='sigmoid')
])

c:\Users\Manindra\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


# compile model

In [16]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [17]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 6, 64)          │        24,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 6, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,025 (144.63 KB)

 Trainable params: 37,025 (144.63 KB)

 Non-trainable params: 0 (0.00 B)

# mlflow tensorflow 

In [18]:
mlflow.tensorflow.autolog(log_models=True, log_input_examples=True)

2026/05/11 15:04:49 WARNING mlflow.utils.autologging_utils: MLflow tensorflow autologging is known to be compatible with 2.16.2 <= tensorflow, but the installed version is 2.16.1. If you encounter errors during autologging, try upgrading / downgrading tensorflow to a compatible version, or try upgrading MLflow.


# early stopping

In [19]:
early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=15,
    restore_best_weights=True
)

# run, fit model with mlflow

In [20]:
with mlflow.start_run(run_name="LSTM_Tuned_run3"):
    
    mlflow.log_param("time_steps", TIMESTEPS)
    mlflow.log_param("model_type", "LSTM_2_Layer")
    
    print("started training")
    
    history = model.fit(
        X_train_seq, y_train_seq,
        epochs=50,
        batch_size=64,
        validation_split=0.2,
        callbacks=[early_stopping],
        verbose=1
    )
    
    test_loss, test_acc = model.evaluate(X_test_seq, y_test_seq, verbose=0)
    
    mlflow.log_metric("test_loss", test_loss)
    mlflow.log_metric("test_accuracy", test_acc)
    
    print(f"\n--- Training Complete ---")
    print(f"Test Accuracy: {test_acc:.4f}")

started training


Epoch 1/50
531/536 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5032 - loss: 0.6949

536/536 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.5033 - loss: 0.6949 - val_accuracy: 0.5114 - val_loss: 0.6929
Epoch 2/50
535/536 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5074 - loss: 0.6930

536/536 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.5074 - loss: 0.6930 - val_accuracy: 0.5119 - val_loss: 0.6928
Epoch 3/50
528/536 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5117 - loss: 0.6925

536/536 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.5118 - loss: 0.6925 - val_accuracy: 0.5130 - val_loss: 0.6926
Epoch 4/50
536/536 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.5175 - loss: 0.6921 - val_accuracy: 0.5225 - val_loss: 0.6927
Epoch 5/50
526/536 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5157 - loss: 0.6928

...

536/536 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.5350 - loss: 0.6886 - val_accuracy: 0.5188 - val_loss: 0.6917
Epoch 46/50
536/536 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.5361 - loss: 0.6886 - val_accuracy: 0.5202 - val_loss: 0.6919
Epoch 47/50
536/536 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.5396 - loss: 0.6890 - val_accuracy: 0.5185 - val_loss: 0.6920
Epoch 48/50
536/536 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.5324 - loss: 0.6890 - val_accuracy: 0.5193 - val_loss: 0.6920
Epoch 49/50
536/536 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.5413 - loss: 0.6882 - val_accuracy: 0.5185 - val_loss: 0.6920
Epoch 50/50
536/536 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.5355 - loss: 0.6879 - val_accuracy: 0.5187 - val_loss: 0.6918
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 247ms/step


2026/05/11 15:07:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/11 15:07:48 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Manindra\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:415: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 12 variables whereas the saved optimizer has 22 variables. "


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 319ms/step


2026/05/11 15:07:48 INFO mlflow.models.model: Found the following environment variables used during model inference: [BYBIT_API_KEY, POSTHOG_API_KEY]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.



--- Training Complete ---
Test Accuracy: 0.5130
🏃 View run LSTM_Tuned_run3 at: http://localhost:5000/#/experiments/13/runs/910f2f41214f4d87bd63dbe86a7200a0
🧪 View experiment at: http://localhost:5000/#/experiments/13
